# Pursuit–evasion: six presentation videos

For each metric (Euclidean, TTC and DCE 1 s), this notebook automatically selects two reproducible initial states on the central-speed slice: one with positive BRT and one with negative BRT. It validates the outcome before creating the six MP4 videos.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display

# The notebook is intended for the repository's examples/ directory.
project_root = Path.cwd().resolve()
if project_root.name == "examples":
    project_root = project_root.parent
if not (project_root / "scripts" / "simulate_pursuit_evasion.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or from examples/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import scripts.simulate_pursuit_evasion as sim

print("Project root:", project_root)

Project root: /home/alessio-grisorio/TESI/hj_reachability


In [9]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
BRT_FILES = {
    "euclidean": "euclidean.npz",
    "ttc": "ttc.npz",
    "dce1": "dce1s.npz",
}

SEEDS = {
    ("euclidean", "positive"): 1101,
    ("euclidean", "negative"): 1102,
    ("ttc", "positive"): 2201,
    ("ttc", "negative"): 2202,
    ("dce1", "positive"): 3301,
    ("dce1", "negative"): 3302,
}

POSITIVE_THRESHOLD = 0.20
NEGATIVE_THRESHOLD = -0.20
MINIMUM_INITIAL_TERMINAL_VALUE = 0.10
GRID_BORDER_NODES_TO_EXCLUDE = 1
MAX_CANDIDATES_TO_TEST = 1000

# Keep the simulation consistent with the BRT horizon.
sim.BRT_HORIZON = 3.0
sim.MAX_SIMULATION_TIME = 3.0
sim.DT = 0.05
sim.SAVE_RESULTS = False
sim.SAVE_STATIC_FIGURE = False
sim.SHOW_STATIC_FIGURE = False
sim.SAVE_ANIMATION = True
sim.SHOW_ANIMATION_WINDOW = False
sim.ANIMATION_FORMAT = "mp4"
sim.ANIMATION_FRAME_STRIDE = 1
sim.ANIMATION_FPS = 20
sim.ANIMATION_DPI = 120

for metric_name, filename in BRT_FILES.items():
    path = project_root / "results" / "brt" /filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing BRT for {metric_name}: {path}")

print("All three BRT files were found.")

All three BRT files were found.


In [11]:
def central_node_index(coordinates):
    """Return the grid node nearest the midpoint of a coordinate interval."""
    values = np.asarray(coordinates, dtype=float)
    midpoint = 0.5 * (values[0] + values[-1])
    return int(np.argmin(np.abs(values - midpoint)))


def candidate_states_on_central_speed_slice(brt_data, sign, seed):
    """Build a deterministic candidate list with v_H and v_E on central nodes."""
    grid = brt_data["grid"]
    brt = np.asarray(brt_data["BRT"])
    terminal = np.asarray(brt_data["V0"])

    i_v_h = central_node_index(grid.coordinate_vectors[3])
    i_v_e = central_node_index(grid.coordinate_vectors[5])
    brt_slice = brt[:, :, :, i_v_h, :, i_v_e]
    terminal_slice = terminal[:, :, :, i_v_h, :, i_v_e]

    mask = terminal_slice > MINIMUM_INITIAL_TERMINAL_VALUE
    if sign == "positive":
        mask &= brt_slice >= POSITIVE_THRESHOLD
    elif sign == "negative":
        mask &= brt_slice <= NEGATIVE_THRESHOLD
    else:
        raise ValueError("sign must be 'positive' or 'negative'.")

    border = GRID_BORDER_NODES_TO_EXCLUDE
    if border > 0:
        interior = np.zeros_like(mask, dtype=bool)
        interior[border:-border, border:-border, border:-border, border:-border] = True
        mask &= interior

    candidates = np.argwhere(mask)
    if len(candidates) == 0:
        raise RuntimeError(
            f"No {sign} candidate exists on the central-speed slice. "
            "Relax the thresholds if necessary."
        )

    # Ordina i candidati privilegiando:
    # 1. distanza dai bordi della griglia;
    # 2. valore BRT nettamente positivo/negativo.
    candidate_values = brt_slice[tuple(candidates.T)]

    shape = np.asarray(brt_slice.shape, dtype=float)
    normalized_indices = candidates / (shape - 1.0)

    distance_from_boundary = np.min(
        np.minimum(
            normalized_indices,
            1.0 - normalized_indices,
        ),
        axis=1,
    )

    # Piccola perturbazione deterministica per evitare che punti
    # equivalenti vengano scelti sempre nello stesso ordine.
    rng = np.random.default_rng(seed)
    tie_breaker = rng.random(len(candidates)) * 1e-9

    if sign == "positive":
        # Prima i punti interni e con BRT più positivo.
        order = np.lexsort(
            (
                tie_breaker,
                -candidate_values,
                -distance_from_boundary,
            )
        )
    else:
        # Prima i punti interni e con BRT più negativo.
        order = np.lexsort(
            (
                tie_breaker,
                candidate_values,
                -distance_from_boundary,
            )
        )

    candidates = candidates[order]

    states = []
    for i_x, i_y, i_theta, i_delta in candidates[:MAX_CANDIDATES_TO_TEST]:
        full_index = (i_x, i_y, i_theta, i_v_h, i_delta, i_v_e)
        state = np.array([
            float(grid.coordinate_vectors[dimension][full_index[dimension]])
            for dimension in range(6)
        ])
        states.append(state)

    return states


def outcome_is_suitable(result, sign):
    """Check whether the simulation clearly illustrates the requested case."""

    collision = (
        result["stop_reason"]
        == "terminal set reached"
    )

    if sign == "negative":
        return (
            collision
            and result["collision_time"] is not None
        )

    # Per il caso positivo accettiamo sia:
    # - il raggiungimento dei 3 secondi senza collisione;
    # - l'uscita dalla griglia senza collisione.
    return not collision


def select_and_validate_case(brt_data, metric_name, sign):
    """Test deterministic candidates until the requested visible outcome is obtained."""
    candidates = candidate_states_on_central_speed_slice(
        brt_data=brt_data,
        sign=sign,
        seed=SEEDS[(metric_name, sign)],
    )
    for attempt, initial_state in enumerate(candidates, start=1):
        result = sim.simulate(brt_data=brt_data, initial_state=initial_state)
        if outcome_is_suitable(result, sign):
            result.update(sim.reconstruct_absolute_trajectories(result, brt_data))
            return initial_state, result, attempt

    raise RuntimeError(
        f"No validated {sign} case found for {metric_name} after "
        f"{len(candidates)} attempts. Increase MAX_CANDIDATES_TO_TEST or relax the thresholds."
    )

In [12]:
# -----------------------------------------------------------------------------
# Select and validate all six simulations before rendering the videos.
# -----------------------------------------------------------------------------
cases = {}

for metric_name, filename in BRT_FILES.items():
    brt_path = project_root / "results" / "brt" / filename
    print(f"\nLoading {metric_name}: {brt_path.name}")
    brt_data = sim.load_saved_brt(brt_path)

    for sign in ("positive", "negative"):
        initial_state, result, attempts = select_and_validate_case(
            brt_data=brt_data, metric_name=metric_name, sign=sign
        )
        cases[(metric_name, sign)] = {
            "brt_data": brt_data,
            "initial_state": initial_state,
            "result": result,
        }
        print(
            f"  {sign:8s} | attempts={attempts:3d} | "
            f"V(-3,x0)={result['brt_value'][0]: .4f} | "
            f"V0(x0)={result['terminal_value'][0]: .4f} | "
            f"v_H={initial_state[3]:.3f} m/s | v_E={initial_state[5]:.3f} m/s | "
            f"stop={result['stop_reason']} at t={result['time'][-1]:.3f} s"
        )

print("\nAll six cases were validated.")


Loading euclidean: euclidean.npz
  positive | attempts=  1 | V(-3,x0)= 0.4795 | V0(x0)= 1.4789 | v_H=5.286 m/s | v_E=5.286 m/s | stop=state left grid at t=0.600 s
  negative | attempts=106 | V(-3,x0)=-3.5268 | V0(x0)= 0.3533 | v_H=5.286 m/s | v_E=5.286 m/s | stop=terminal set reached at t=0.375 s

Loading ttc: ttc.npz
  positive | attempts=  1 | V(-3,x0)= 1.7388 | V0(x0)= 6.0000 | v_H=5.286 m/s | v_E=5.286 m/s | stop=grid boundary reached at t=0.612 s
  negative | attempts=  1 | V(-3,x0)=-2.8817 | V0(x0)= 0.1153 | v_H=5.286 m/s | v_E=5.286 m/s | stop=terminal set reached at t=0.004 s

Loading dce1: dce1s.npz
  positive | attempts=  1 | V(-3,x0)= 1.3011 | V0(x0)= 2.4590 | v_H=5.286 m/s | v_E=5.286 m/s | stop=grid boundary reached at t=0.612 s
  negative | attempts=  1 | V(-3,x0)=-1.5698 | V0(x0)= 0.5200 | v_H=5.286 m/s | v_E=5.286 m/s | stop=terminal set reached at t=0.298 s

All six cases were validated.


In [13]:
# -----------------------------------------------------------------------------
# Render, save and display the six videos with the script's existing layout.
# -----------------------------------------------------------------------------
video_paths = {}

for metric_name in BRT_FILES:
    for sign in ("positive", "negative"):
        case = cases[(metric_name, sign)]
        result = case["result"]

        print(f"\nCreating video: {metric_name} — BRT {sign}")
        figure, simulation_animation, generated_path = sim.create_animation(result=result)
        plt.close(figure)

        final_path = generated_path.with_name(f"{metric_name}_brt_{sign}.mp4")
        if final_path.exists():
            final_path.unlink()
        generated_path.replace(final_path)
        video_paths[(metric_name, sign)] = final_path

        print("Saved in:", final_path.resolve())
        display(Video(filename=str(final_path), embed=True))

print("\nSix videos completed:")
for key, path in video_paths.items():
    print(f"  {key[0]:9s} {key[1]:8s} -> {path.name}")


Creating video: euclidean — BRT positive
Saved in: /home/alessio-grisorio/TESI/hj_reachability/results/animations/euclidean_brt_positive.mp4



Creating video: euclidean — BRT negative
Saved in: /home/alessio-grisorio/TESI/hj_reachability/results/animations/euclidean_brt_negative.mp4



Creating video: ttc — BRT positive
Saved in: /home/alessio-grisorio/TESI/hj_reachability/results/animations/ttc_brt_positive.mp4



Creating video: ttc — BRT negative
Saved in: /home/alessio-grisorio/TESI/hj_reachability/results/animations/ttc_brt_negative.mp4



Creating video: dce1 — BRT positive
Saved in: /home/alessio-grisorio/TESI/hj_reachability/results/animations/dce1_brt_positive.mp4



Creating video: dce1 — BRT negative
Saved in: /home/alessio-grisorio/TESI/hj_reachability/results/animations/dce1_brt_negative.mp4



Six videos completed:
  euclidean positive -> euclidean_brt_positive.mp4
  euclidean negative -> euclidean_brt_negative.mp4
  ttc       positive -> ttc_brt_positive.mp4
  ttc       negative -> ttc_brt_negative.mp4
  dce1      positive -> dce1_brt_positive.mp4
  dce1      negative -> dce1_brt_negative.mp4


In [14]:
# ============================================================================
# Rigenerazione del solo video TTC con BRT negativo
# ============================================================================

metric_name = "ttc"
sign = "negative"

MIN_INITIAL_V0 = 0.10
MIN_VISIBLE_COLLISION_TIME = 0.15
MAX_ATTEMPTS = 1000

brt_path = (
    project_root
    / "results"
    / "brt"
    / BRT_FILES[metric_name]
)

print("Loading:", brt_path)

brt_data = sim.load_saved_brt(brt_path)

# Genera candidati riproducibili con:
# - BRT negativo;
# - velocità sui nodi centrali della griglia.
candidate_states = candidate_states_on_central_speed_slice(
    brt_data=brt_data,
    sign=sign,
    seed=9876,  # seed diverso da quello usato precedentemente
)

selected_state = None
selected_result = None

for attempt, initial_state in enumerate(
    candidate_states[:MAX_ATTEMPTS],
    start=1,
):
    # Controllo esplicito del valore terminale iniziale
    initial_game = sim.evaluate_game(
        brt_data=brt_data,
        state=initial_state,
        time=0.0,
    )

    initial_v0 = initial_game["terminal_value"]
    initial_brt = initial_game["brt_value"]

    # Lo stato deve essere inizialmente fuori dal terminal set
    if initial_v0 <= MIN_INITIAL_V0:
        continue

    result = sim.simulate(
        brt_data=brt_data,
        initial_state=initial_state,
    )

    collision_time = result["collision_time"]

    valid_collision = (
        result["stop_reason"] == "terminal set reached"
        and collision_time is not None
        and collision_time >= MIN_VISIBLE_COLLISION_TIME
    )

    if valid_collision:
        selected_state = initial_state
        selected_result = result
        break

    if attempt % 100 == 0:
        print(
            f"Tentativo {attempt}: "
            f"BRT={initial_brt:.4f}, "
            f"V0={initial_v0:.4f}, "
            f"stop={result['stop_reason']}, "
            f"t={result['time'][-1]:.3f} s"
        )

if selected_result is None:
    raise RuntimeError(
        "Non è stato trovato un caso TTC negativo adatto. "
        "Prova ad aumentare MAX_ATTEMPTS oppure a ridurre "
        "MIN_VISIBLE_COLLISION_TIME."
    )

# Ricostruzione delle traiettorie assolute necessarie all'animazione
selected_result.update(
    sim.reconstruct_absolute_trajectories(
        result=selected_result,
        brt_data=brt_data,
    )
)

print("\nCaso TTC negativo selezionato")
print("Tentativi:", attempt)
print("Stato iniziale:", selected_state)
print("BRT iniziale:", selected_result["brt_value"][0])
print("V0 iniziale:", selected_result["terminal_value"][0])
print("Tempo di collisione:", selected_result["collision_time"])
print("v_H iniziale:", selected_state[3], "m/s")
print("v_E iniziale:", selected_state[5], "m/s")

# Generazione della sola animazione TTC negativa
figure, simulation_animation, generated_path = sim.create_animation(
    result=selected_result
)

plt.close(figure)

final_path = generated_path.with_name(
    "ttc_brt_negative.mp4"
)

# Sostituisce il precedente video TTC negativo
if final_path.exists():
    final_path.unlink()

generated_path.replace(final_path)

# Aggiorna anche il dizionario del notebook
cases[(metric_name, sign)] = {
    "brt_data": brt_data,
    "initial_state": selected_state,
    "result": selected_result,
}

video_paths[(metric_name, sign)] = final_path

print("\nNuovo video salvato in:")
print(final_path.resolve())

display(
    Video(
        filename=str(final_path),
        embed=True,
    )
)

Loading: /home/alessio-grisorio/TESI/hj_reachability/results/brt/ttc.npz

Caso TTC negativo selezionato
Tentativi: 4
Stato iniziale: [ 3.          2.         -0.22439948  5.28571415 -0.02617992  5.28571415]
BRT iniziale: -2.7027816772460938
V0 iniziale: 0.12688474357128143
Tempo di collisione: 0.1779505400772914
v_H iniziale: 5.285714149475098 m/s
v_E iniziale: 5.285714149475098 m/s

Nuovo video salvato in:
/home/alessio-grisorio/TESI/hj_reachability/results/animations/ttc_brt_negative.mp4
